In [ ]:
!git clone https://github.com/Harindhar10/DeepRetro.git

In [2]:
# load up USPTO-50k test dataset
import pandas as pd

df = pd.read_csv("/kaggle/working/DFS/DeepRetro/data/uspto_50k_test_250.csv")
df.head()

,input,output,reaction_type,cluster_id
0,CS(=O)c1cccc(-c2nc(C=O)ccc2OCCO[Si](C)(C)C(C)(...,CC(C)(C)[Si](C)(C)OCCOc1ccc(C=O)nc1Br.CS(=O)c1...,3,0
1,CC(C)=CCSc1ccc(Br)cc1,CC(C)=CCBr.Sc1ccc(Br)cc1,1,0
2,CCC1(c2ccc(C=O)s2)OCCO1,CCC1(c2cccs2)OCCO1.CN(C)C=O,3,0
3,O=C1Nc2ccccc2C1c1cc(Br)ccc1O,O=C1Nc2ccccc2C1(O)c1cc(Br)ccc1O,9,0
4,CCCCCCCCCCCCCCCCOCC(CN)CC#N,CCCCCCCCCCCCCCCCOCC(CC#N)CN=[N+]=[N-],9,0


In [9]:
mol1 = df.iloc[0]['input']
trg1 = df.iloc[0]['output']

In [ ]:
USER_PROMPT = """You are an expert organic chemist specializing in retrosynthesis. When given a target molecule, you will perform a single-step retrosynthesis, providing 3-5 possible precursor molecules or reactions that could lead to the formation of the target molecule. 

Present your final analysis in a specific JSON format. For each suggestion, provide the precursor molecules in SMILES notation and a brief explanation of the reaction type and any key conditions or reagents needed. Use standard organic chemistry notation and terminology in your explanations. 

If the molecule is too simple for meaningful retrosynthesis, state this in a single JSON object with an appropriate explanation.

Perform a single-step retrosynthesis on the following molecule, providing 3-5 possible precursors or reactions:



Present your final analysis in the following JSON format:

<json>
{
  "data": [
    [precursor1_SMILES, precursor2_SMILES, ...],
    [precursor1_SMILES, precursor2_SMILES, ...],
    ...
  ],
  "explanation": [
    "explanation 1",
    "explanation 2",
    ...
  ],
  "confidence_scores": [
    confidence_score1,
    confidence_score2,
    ...
  ]
}
</json>

For each suggestion in the "data" array, provide the precursor molecules in SMILES notation. Ensure to provide only valid SMILES strings.

In the corresponding "explanation" array, briefly explain the reaction type and any key conditions or reagents needed.

In the "confidence_scores" array, provide a confidence score for each suggestion between 0 and 1, indicating your confidence in the proposed retrosynthesis pathway.

Ensure that the number of entries in "data", "explanation", and "confidence_scores" are the same.
"""

In [12]:
USER_PROMPT

'You are an expert organic chemist specializing in retrosynthesis. When given a target molecule, you will perform a single-step retrosynthesis, providing 3-5 possible precursor molecules or reactions that could lead to the formation of the target molecule. \n\nPresent your final analysis in a specific JSON format. For each suggestion, provide the precursor molecules in SMILES notation and a brief explanation of the reaction type and any key conditions or reagents needed. Use standard organic chemistry notation and terminology in your explanations. \n\nIf the molecule is too simple for meaningful retrosynthesis, state this in a single JSON object with an appropriate explanation.\n\nPerform a single-step retrosynthesis on the following molecule, providing 3-5 possible precursors or reactions:\n\n{mol1}\n\nPresent your final analysis in the following JSON format:\n\n<json>\n{\n  "data": [\n    [precursor1_SMILES, precursor2_SMILES, ...],\n    [precursor1_SMILES, precursor2_SMILES, ...],\n

# One-step retrosynthesis evaluation of a causal LM

Set `MODEL_ID` in the config cell below to any HuggingFace causal LM; everything downstream
(prompting, generation, checkpoint paths, scoring) is derived from it.

Scored with the same protocol as `DeepRetro/notebooks/prod_challenge.ipynb` (cells 31-39):
canonicalize predicted precursors and ground-truth reactants with RDKit, then require equal
cardinality plus set membership.

- `all_correct` — every predicted precursor is in the ground truth, and the counts match.
- `any_correct` — at least one predicted precursor is in the ground truth, and the counts match.

Reported at **top-1** (first proposal only, directly comparable to prod_challenge, which scores
only `steps[0]`) and at **top-k** (a hit in any of the 3-5 proposals the model returns).

In [3]:
!pip install -q rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 31.4 MB/s eta 0:00:00:00:0100:01


In [ ]:
!pip install ipy

In [ ]:
import ast
import json
import os
import re
import sys

import pandas as pd
import torch
from rdkit import Chem, RDLogger
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

RDLogger.DisableLog("rdApp.*")  # silence the parse errors we already handle

REPO = "/kaggle/working/DFS/DeepRetro"
sys.path.insert(0, REPO)

# Reuse DeepRetro's own prompts. SYS/USER_PROMPT_OPENAI are the non-CoT pair the repo
# uses for models that do not emit <cot> tags -- the right family for instruct models
# like Olmo-3-Instruct. src/variables.py is pure string constants, so this import pulls
# in no litellm/langfuse.
from src.variables import SYS_PROMPT_OPENAI, USER_PROMPT_OPENAI

# --- the only knob that has to change to evaluate a different model ---
MODEL_ID = "allenai/Olmo-3-7B-Instruct"

DATA_CSV = f"{REPO}/data/USPTO-50k_500.csv"

# Output paths are derived from MODEL_ID so two models never share a checkpoint file.
# run_generation() resumes from RAW_JSONL, so a shared path would make a new model
# silently inherit the previous model's completions.
MODEL_SLUG = MODEL_ID.rstrip("/").split("/")[-1]
DATA_SLUG = os.path.splitext(os.path.basename(DATA_CSV))[0]
OUT_DIR = f"{REPO}/results/{DATA_SLUG}_{MODEL_SLUG}"
RAW_JSONL = f"{OUT_DIR}/raw_generations.jsonl"
RESULTS_CSV = f"{OUT_DIR}/results.csv"
os.makedirs(OUT_DIR, exist_ok=True)
print("writing to", OUT_DIR)

df_eval = pd.read_csv(DATA_CSV, index_col=0)  # index is mol_no
print(df_eval.shape, df_eval.columns.tolist())
df_eval.head()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # required for correct batched generation

# T4s are Turing: fp16 only, no bf16. device_map="auto" shards a 7B across both cards.
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,  # transformers>=5 spelling; torch_dtype is deprecated
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
print(MODEL_ID, "->", model.dtype, getattr(model, "hf_device_map", model.device))


def build_prompt(molecule: str) -> str:
    """Render the chat prompt exactly as src/utils/llm.py::call_LLM builds its messages.

    Note: .replace() rather than .format() -- USER_PROMPT_OPENAI embeds a literal JSON
    schema with { } braces, so .format() would raise.

    Base models ship no chat template, so fall back to a plain system+user concatenation
    rather than crashing.
    """
    user = USER_PROMPT_OPENAI.replace("{target_smiles}", molecule)
    if getattr(tokenizer, "chat_template", None):
        messages = [
            {"role": "system", "content": SYS_PROMPT_OPENAI},
            {"role": "user", "content": user},
        ]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    return f"{SYS_PROMPT_OPENAI}\n\n{user}\n\n"


print(build_prompt(df_eval.iloc[0]["input"])[:1200])

In [ ]:
# Ports of src/utils/llm.py::split_json_openAI and ::validate_split_json.
# Inlined rather than imported: src.utils.llm pulls in litellm/langfuse, which are not
# installed here. Two robustness additions over the originals, because a 7B model drops
# the <json> tags and emits real JSON far more often than Claude/o1 do:
#   - json.loads first, ast.literal_eval as fallback (the repo uses only the latter)
#   - regex fallback to a bare {...} block when the tags are missing

_BARE_JSON = re.compile(r"\{.*\"data\".*\}", re.DOTALL)


def split_json_content(res_text: str):
    """Return (status, json_content). 200 on success, 502 on failure."""
    start = res_text.find("<json>")
    end = res_text.find("</json>")
    if start != -1 and end != -1 and end > start:
        content = res_text[start + len("<json>"):end].strip()
        if content:
            return 200, content
    m = _BARE_JSON.search(res_text)  # tags missing -- salvage the object itself
    if m:
        return 200, m.group(0).strip()
    return 502, ""


def loads_lenient(json_content: str):
    try:
        return json.loads(json_content)
    except Exception:
        return ast.literal_eval(json_content)  # Python-literal style (repo behaviour)


def validate_split_json(json_content: str):
    """Return (status, molecules, explanations, confidence_scores). 504 on failure."""
    try:
        result = loads_lenient(json_content)
        return 200, result["data"], result["explanation"], result["confidence_scores"]
    except Exception:
        return 504, [], [], []


def parse_response(res_text: str):
    """Full response -> (status, proposals). proposals is a list of precursor-SMILES lists."""
    status, json_content = split_json_content(res_text)
    if status != 200:
        return status, []
    status, molecules, _expl, _conf = validate_split_json(json_content)
    if status != 200:
        return status, []
    # Normalise: a single flat list of strings means one proposal, not many.
    if molecules and all(isinstance(m, str) for m in molecules):
        molecules = [molecules]
    proposals = [
        [s for s in prop if isinstance(s, str) and s.strip()]
        for prop in molecules
        if isinstance(prop, list)
    ]
    proposals = [p for p in proposals if p]
    return (200, proposals) if proposals else (504, [])

In [ ]:
MAX_NEW_TOKENS = 1024
BATCH_SIZE = 4  # set to 1 if the two T4s run out of memory


@torch.inference_mode()
def generate_batch(molecules):
    """Greedy-decode one batch; returns the completions with the prompt stripped."""
    prompts = [build_prompt(m) for m in molecules]
    # add_special_tokens=False: chat templates already emit BOS themselves.
    enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    out = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,  # greedy, matching the repo's temperature=0.0
        pad_token_id=tokenizer.pad_token_id,
    )
    gen = out[:, enc["input_ids"].shape[1]:]
    return tokenizer.batch_decode(gen, skip_special_tokens=True)


def load_checkpoint(path):
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    rec = json.loads(line)
                    done[rec["mol_no"]] = rec
    return done


def run_generation(df, limit=None):
    """Generate completions for df, appending to RAW_JSONL and skipping finished rows."""
    done = load_checkpoint(RAW_JSONL)
    todo = [(i, row) for i, row in df.iterrows() if int(i) not in done]
    if limit is not None:
        todo = todo[:limit]
    print(f"{len(done)} already done, generating {len(todo)}")

    with open(RAW_JSONL, "a") as fout:
        for start in tqdm(range(0, len(todo), BATCH_SIZE)):
            chunk = todo[start:start + BATCH_SIZE]
            try:
                texts = generate_batch([row["input"] for _, row in chunk])
            except Exception as e:
                print(f"generation failed for {[int(i) for i, _ in chunk]}: {e}")
                texts = [""] * len(chunk)
            for (i, row), text in zip(chunk, texts):
                fout.write(json.dumps({
                    "mol_no": int(i),
                    "model_id": MODEL_ID,
                    "input": row["input"],
                    "output": row["output"],
                    "raw": text,
                }) + "\n")
            fout.flush()  # checkpoint survives a kernel restart mid-run


# Smoke test: one molecule end to end before committing to all 500.
_smoke = generate_batch([df_eval.iloc[0]["input"]])[0]
print(_smoke[:2000])
print("\n--- parsed ---")
print(parse_response(_smoke))

In [ ]:
# Dry run on the first 10 rows first. Re-running this cell should say "10 already done,
# generating 0" -- that confirms the checkpoint works before the long run.
run_generation(df_eval, limit=10)

In [ ]:
# Full run over all 500 molecules. Hours on 2x T4 -- resumable, so it is safe to
# interrupt and re-run this cell.
run_generation(df_eval)

In [ ]:
# The prod_challenge.ipynb metric (cell 38), factored so top-1 and top-k share it.
# One deviation from the original: CanonSmiles is wrapped in try/except. prod_challenge
# lets it throw because AiZynthFinder only ever emitted valid SMILES; an LLM will not, and
# an invalid proposal has to score 0 rather than abort the loop.


def canon(smiles: str):
    try:
        return Chem.CanonSmiles(smiles)
    except Exception:
        return None


def score_proposal(pred_smiles_list, gt_canon):
    """Return (any_correct, all_correct, not_common, missed, valid)."""
    pred = [canon(s) for s in pred_smiles_list]
    if any(p is None for p in pred):
        return 0, 0, [s for s, p in zip(pred_smiles_list, pred) if p is None], gt_canon, False
    same_len = len(gt_canon) == len(pred)
    any_c = int(any(p in gt_canon for p in pred) and same_len)
    all_c = int(all(p in gt_canon for p in pred) and same_len)
    not_common = [p for p in pred if p not in gt_canon]
    missed = [g for g in gt_canon if g not in pred]
    return any_c, all_c, not_common, missed, True


def score_record(rec):
    """Score one generation record at top-1 and top-k."""
    gt_canon = [canon(s) for s in str(rec["output"]).split(".")]
    gt_canon = [g for g in gt_canon if g is not None]

    status, proposals = parse_response(rec["raw"])
    row = {
        "mol_no": rec["mol_no"],
        "input": rec["input"],
        "output": rec["output"],
        "parse_status": status,
        "n_proposals": len(proposals),
        "proposals": proposals,
        "any_correct": 0,
        "all_correct": 0,
        "any_correct_topk": 0,
        "all_correct_topk": 0,
        "first_hit_rank": None,
        "not_common": [],
        "missed": gt_canon,
        "has_invalid_smiles": 0,
    }
    if status != 200 or not proposals:
        return row  # parse failure counts as incorrect, it is not dropped

    any_invalid = False
    for rank, prop in enumerate(proposals, start=1):
        any_c, all_c, not_common, missed, valid = score_proposal(prop, gt_canon)
        if not valid:
            any_invalid = True
        if rank == 1:  # top-1: exactly what prod_challenge scores
            row.update(any_correct=any_c, all_correct=all_c,
                       not_common=not_common, missed=missed)
        row["any_correct_topk"] = max(row["any_correct_topk"], any_c)
        if all_c and row["first_hit_rank"] is None:
            row["first_hit_rank"] = rank
        row["all_correct_topk"] = max(row["all_correct_topk"], all_c)
    row["has_invalid_smiles"] = int(any_invalid)
    return row


# Sanity check: the ground truth must score (1, 1) against itself on every row.
_gt_ok = all(
    score_proposal([s for s in str(r["output"]).split(".")],
                   [canon(s) for s in str(r["output"]).split(".")])[:2] == (1, 1)
    for _, r in df_eval.iterrows()
)
print("ground truth scores perfectly against itself:", _gt_ok)

In [ ]:
records = load_checkpoint(RAW_JSONL)
df_results = pd.DataFrame([score_record(rec) for rec in records.values()])
df_results = df_results.set_index("mol_no").sort_index()
df_results.to_csv(RESULTS_CSV)

n = len(df_results)
print(f"model: {MODEL_ID}")
print(f"Scored {n} / {len(df_eval)} molecules from {RAW_JSONL}\n")

# prod_challenge cell 39 wording, so the numbers sit side by side with the DeepRetro run.
print("=== top-1 (first proposal -- prod_challenge protocol) ===")
print("All correct %", 100 * df_results.all_correct.sum() / n)
print("Any correct %", 100 * df_results.any_correct.sum() / n)

print("\n=== top-k (best of the 3-5 proposals) ===")
print("All correct %", 100 * df_results.all_correct_topk.sum() / n)
print("Any correct %", 100 * df_results.any_correct_topk.sum() / n)

print("\n=== top-n curve (all_correct) ===")
for k in (1, 3, 5):
    hits = df_results.first_hit_rank.dropna().le(k).sum()
    print(f"top-{k}: {100 * hits / n:.2f}%")

print("\n=== response quality ===")
print(f"parse failures  : {100 * (df_results.parse_status != 200).sum() / n:.2f}%")
print(f"invalid SMILES  : {100 * df_results.has_invalid_smiles.sum() / n:.2f}%")
print(f"mean proposals  : {df_results.n_proposals.mean():.2f}")
print(f"\nDenominator is all {n} scored rows -- parse failures count as incorrect. "
      "prod_challenge dropna()'d its pipeline failures (cell 36), so its numbers are "
      "computed over a smaller, easier set.")

df_results.head(10)